In [1]:
import torch
from torchvision.io import read_image, ImageReadMode
from torchvision.transforms.functional import crop
import numpy as np
import os
from utils import get_base_points, get_oriented_points_and_views
from tqdm import tqdm
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
class Dataset:
    def __init__(self, folder, nb_valid=4, seed=-1, **kwargs):
        self.width = 0
        self.height = 0
        self.px_width = 0
        self.px_height = 0
        self.orig_px_width = 0
        self.orig_px_height = 0
        self.point_min = (0, 0, 0)
        self.point_max = (100,100,100)
        self.slices = []
        self.slices_valid = []
        self.X, self.Y = ([],[])
        self.name = kwargs.get('name', os.path.basename(os.path.normpath(folder)))
        self.has_gt = False
        self.roi_2d = None
        self.image_stpe = kwargs.get('image_step', 1)

        prefix = kwargs.get("prefix","us/img_")
        suffix = kwargs.get("suffix",".jpg")
        reverse_quat = kwargs.get("reverse_quat",False)
        self.exclude_valid = kwargs.get("exclude_valid", True)

        image_buffer = []
        gt_buffer = []
        points_buffer = []
        points_numpy = []
        

In [7]:
from torchvision.io import read_image, ImageReadMode

In [5]:
folder = "D:\\0-Code\\NeUF\\data\\simu_56\\"

os.path.basename(folder)

''

In [23]:
im = read_image("D:\\0-Code\\NeUF\\data\\simu_56\\us\\us0.jpg")
im.shape

torch.Size([1, 100, 150])

In [25]:
import torch

torch.flip(im, [2]).shape

torch.Size([1, 100, 150])

In [36]:
np.tile(np.arange(6).reshape(-1, 2), 3)


array([[0, 1, 0, 1, 0, 1],
       [2, 3, 2, 3, 2, 3],
       [4, 5, 4, 5, 4, 5]])

In [37]:
np.repeat(np.arange(6).reshape(-1, 2), 3)

array([0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5])

In [27]:
torch.flip(im, [2])

tensor([[[85, 58, 78,  ..., 86, 68, 91],
         [53, 32, 57,  ..., 25, 31, 80],
         [76, 38, 49,  ...,  6,  0, 34],
         ...,
         [64, 29, 53,  ..., 62, 42, 77],
         [60, 26, 51,  ..., 38, 20, 63],
         [56, 22, 50,  ..., 46, 35, 89]]], dtype=torch.uint8)

In [ ]:
def get_base_point(width, height, width_px, height_px):
    
    pixel_size_W = width / width_px
    pixel_size_H = height / height_px

    start_x = - width / 2 + pixel_size_W / 2
    start_y = 0 + pixel_size_H / 2

    X = np.tile(np.arange(0, width_px, 1), width_px) * pixel_size_W + start_x
    Y = np.repeat(np.arange(0, height_px, 1), height_px) * pixel_size_H + start_y



SyntaxError: expected ':' (446557765.py, line 1)

In [ ]:
def get_oriented_points_and_views(X_base_points, Y_base_points, position, rotation):
    width_points = X_base_points
    heigh_point = Y_base_points

    probe_points = np.stack((heigh_point, np.zeros(width_points), width_points), axis = 1)

    points = np.apply_along_axis(rotation.apply, 1, probe_points + position)
    viewdirs = np.apply_along_axis(rotation.apply, 1, np.stack((-np.ones(heigh_point), np.zeros(heigh_point), np.zeros(heigh_point), axis=1)))

In [ ]:
def query(self, input, dirs, net_chunk = 1024 * 64):
    